# LLM Fine-Tuning Deep Dive, Part 2 of 3: Parameter-Based Techniques + QLoRA & Quantization

> **This is Part 2 of a three-notebook fine-tuning arc:**
>
> 1. [Part 1: Data-based techniques](01-llm-finetuning-data-techniques.ipynb) asks what Riverside wants the model to learn: catalog language, instruction behavior, or editor preference.
> 2. **Part 2 (this notebook)** asks how much model state must change while learning: all weights, selected layers, or small adapters. It then asks what can be stored more compactly during training or serving.
> 3. [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) asks what each result actually proves, which comparisons are fair, and what a candidate must demonstrate before release.

**Where Part 1 left off:** Riverside produced a full continued-pretraining checkpoint, an instruction-tuned LoRA adapter, and a DPO adapter. Those artifacts changed both the learning objective and, in some cases, the parameter strategy.

Part 2 isolates the **question**, not a causal quality result: keep returning to continued pretraining as the common behavior goal and inspect alternative places to store the update. The teaching runs make trainable state and artifact structure concrete, but their data slices and budgets are not fully matched. Part 3 will explain why that prevents a quality ranking.

## Table of Contents (Part 2)

1. [Setup: Reloading Where Part 1 Left Off](#setup-reloading-where-part-1-left-off)
2. [Parameter-Based Axis: How Much Model State Must Change?](#parameter-based-axis-how-much-model-state-must-change)
   - [Concept 4: Full Fine-Tuning](#concept-4-parameter-based-full-fine-tuning)
   - [Concept 5: Partial Freezing](#concept-5-parameter-based-partial-fine-tuning-layer-freezing)
   - [Concept 6: LoRA](#concept-6-parameter-based-parameter-efficient-fine-tuning-lora)
   - [Concept 7: QLoRA](#concept-7-parameter-based-qlora---put-the-frozen-base-on-a-memory-diet)
   - [Optional Production Extension](#optional-production-extension-shrink-the-artifact-after-training)
   - [Visual Comparison: Parameter Counts Across All Techniques](#visual-comparison-parameter-counts-across-all-techniques)

---

## Setup: Reloading Where Part 1 Left Off

Kernels do not share memory between notebooks. This section reloads the tokenizer, corpus loader, generation helper, baseline model, and Part 1 instruction adapter from disk. Reloading is implementation setup; the parameter-budget story begins after the saved artifacts are reconstructed.

## Fine-Tuning Roadmap: The Objective Is Set, Now Change the Parameter Budget

Part 1 asked **what behavior should be learned?** Part 2 asks a different question:

> If Riverside keeps the continued-pretraining goal, how much of the model must be allowed to change, and what does each choice cost?

The strategies below are alternative ways to carry the same kind of learning signal. They are not stages that every checkpoint must pass through:

```mermaid
flowchart LR
    Goal["Same learning goal<br/>expect Riverside prose"] --> Full["Allow every weight to move"]
    Goal --> Partial["Allow selected upper blocks to move"]
    Goal --> LoRA["Freeze the base<br/>train a small side path"]
    LoRA --> QLoRA["Also store the frozen base compactly"]
    Full --> Compare["Part 3<br/>evaluate evidence"]
    Partial --> Compare
    LoRA --> Compare
```

Only a few quantities are directly comparable in the teaching runs: which state is trainable, the number of trainable parameters, and the structure of the saved artifact. The runs use different genre slices and some different hyperparameters, so their generated outputs do **not** isolate parameter strategy as the cause of a quality difference.

The spectrum image supplies the physical intuition: which weights may move, what remains frozen, and what must still occupy memory.

![Parameter efficiency spectrum comparing full fine-tuning, partial fine-tuning, LoRA, and QLoRA](images/parameter-strategies-spectrum.png)

A frozen model can still consume substantial memory. That observation will motivate QLoRA after ordinary LoRA has made the trainable update small.

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(...).to(device)` loads pretrained weights and moves the model to the training device (CPU/GPU); `model.generate()` inside `torch.no_grad()` runs autoregressive decoding without tracking gradients. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(...)` loads weights, with `tf.device(...)` selecting the device; Keras has no built-in `.generate()` for causal LMs outside HF's `TFGenerationMixin.generate()`, and gradient tracking is simply skipped by not wrapping calls in a `tf.GradientTape()`.

In [ ]:
# Re-establish Part 1's SmolLM2 foundations; Part 1 contains the full data-objective walkthrough.
from pathlib import Path
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."
STORY_SEED = "Aria Voss stared at the signal counting itself out in prime numbers and"
PROMPT = "Continue this fiction narrative in the same style: " + STORY_SEED

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_config = base_model.config
print(
    f"Loaded {MODEL_NAME}: {model_config.num_hidden_layers} decoder layers, "
    f"hidden size {model_config.hidden_size}, "
    f"{sum(p.numel() for p in base_model.parameters()):,} parameters"
)


def format_instruction(prompt, tokenizer_obj=tokenizer):
    """Render one user request with the model's native instruction template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    return tokenizer_obj.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate(model, prompt, max_new_tokens=60, instruction=True, tokenizer_obj=tokenizer):
    """Generate only new tokens, using the native chat template for instruction requests."""
    model.eval()
    model_input = format_instruction(prompt, tokenizer_obj) if instruction else prompt
    model_device = next(model.parameters()).device
    inputs = tokenizer_obj(model_input, return_tensors="pt")
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer_obj.pad_token_id,
            eos_token_id=tokenizer_obj.eos_token_id,
        )
    completion = tokenizer_obj.decode(
        output[0][prompt_len:], skip_special_tokens=True
    ).strip()
    return completion or "[model stopped immediately]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")


In [ ]:
# Corpus loader + the tokenize_causal() helper Concepts 5/6 reuse for training,
# plus every visualization/training/PEFT import this notebook needs.
try:

    # VS Code injects this variable so the notebook can locate its own folder
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:

        # Fall back to the script's own location when run outside VS Code's notebook runtime
        _notebook_dir = Path(__file__).parent
    except NameError:

        # Last resort: assume the current working directory is close enough
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():

    # Handle kernels whose cwd is the repo root instead of this notebook's own folder
    _fallback = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if _fallback.exists():
        CONTENT_DIR = _fallback

print(f"Content directory: {CONTENT_DIR.absolute()}")

NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}


def load_corpus_paragraphs(novels=None, min_len=200):
    """Load qualifying paragraphs from every chapter of the selected novels."""
    if novels is None:

        # Default to every novel in the catalog
        novels = list(NOVELS.keys())
    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:

            # Skip aliases that don't map to a known novel
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():

            # Skip novels whose content folder isn't present on disk
            continue

        # Read every chapter so training is not biased toward the beginnings of novels
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):

                # Collapse embedded newlines so each paragraph is a single line of text
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:

                    # Drop very short fragments that aren't real paragraphs
                    paragraphs.append(para)
    return paragraphs


def tokenize_causal(examples, tokenizer, max_length=128):
    """Tokenize one batched ``Dataset.map`` input for causal language modeling.

    Expected input shape: ``examples == {"text": list[str]}``, where
    ``len(examples["text"]) == B`` for the current batch. Because overflowing text is
    split into chunks, the returned ``input_ids``, ``attention_mask``, and ``labels``
    each have shape ``[B_chunks, max_length]``, where ``B_chunks`` may be greater than
    ``B``.
    """
    # Tokenize the batch, padding/truncating every example to the same length
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length, return_overflowing_tokens=True
    )

    # Labels start as a copy of input_ids, with padding positions masked to -100
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from datasets import Dataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Silence noisy library warnings that aren't actionable in this demo
warnings.filterwarnings("ignore")

# Apply a consistent figure resolution/font size to every plot in this notebook
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

# Apply a consistent seaborn theme/palette to every plot in this notebook
sns.set_theme(style="whitegrid", palette="muted")

print("Corpus loader, tokenize_causal(), and training/visualization imports ready.")

### Reloading Part 1's Instruction-Tuned LoRA Adapter

The later introspection section uses the instruction-tuning adapter produced by Part 1. The base model, tokenizer, target modules, and tensor shapes must match exactly.

> **Checkpoint migration boundary:** existing previous-model checkpoint folders are architecture-incompatible with SmolLM2. This notebook does not modify them. Move or remove those old artifacts, then rerun Part 1 with `MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"` to regenerate `./checkpoints/instruction-lora` and the other Part 1 artifacts before executing the reload below. Then rerun this notebook's training cells to regenerate `partial-freeze` and `peft-lora`.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, adapter_dir)` attaches previously trained LoRA adapter weights on top of a frozen base model, reconstructing the exact adapted model from Part 1's checkpoint. **Keras/TF equivalent:** there's no first-party Keras LoRA/PEFT library — the closest analog is reloading a full model (or a frozen-base-plus-trainable-sublayer subclass) via `model.load_weights(...)`; Keras users would typically just reload the entire fine-tuned model rather than a small swappable adapter.

In [ ]:
# Reload the instruction-tuned adapter only after Part 1 has regenerated it for MODEL_NAME.
instruction_adapter_dir = Path("./checkpoints/instruction-lora")
adapter_config_path = instruction_adapter_dir / "adapter_config.json"
if not adapter_config_path.is_file():
    raise FileNotFoundError(
        f"Missing {adapter_config_path}. Rerun Part 1 with MODEL_NAME={MODEL_NAME!r}."
    )

saved_adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
saved_base = saved_adapter_config.get("base_model_name_or_path")
if saved_base != MODEL_NAME:
    raise RuntimeError(
        f"Checkpoint base {saved_base!r} is incompatible with {MODEL_NAME!r}. "
        "Leave the old artifact untouched and regenerate Part 1 checkpoints."
    )

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, instruction_adapter_dir
).to(device)
instruct_lora_model.eval()
print("Reloaded the SmolLM2 instruction adapter generated by Part 1.")
print(generate(instruct_lora_model, PROMPT))


---

## Parameter-Based Axis: How Much Model State Must Change?

**Riverside's local problem:** the catalog will keep growing, but IT has one modest machine. The model must be retrainable without confusing a smaller update budget with a different learning goal.

Keep the continued-pretraining question fixed: **make Riverside prose less surprising to the model.** Now vary only where that learning is allowed to live.

### Start with the Physical Cost

Training needs more than stored model weights. Every trainable value also needs a gradient and optimizer history, while intermediate activations must remain available for backpropagation. Freezing a value removes its gradient and optimizer state, but the forward pass may still need the value itself.

That gives four increasingly constrained strategies:

| Strategy | What may learn? | What still occupies memory? | Artifact produced by this arc | Honest conclusion available here |
| --- | --- | --- | --- | --- |
| Full fine-tuning | Every model weight | Full model, activations, gradients, and optimizer state for all weights | Complete checkpoint | Maximum update freedom and largest trainable-state budget |
| Partial freezing | Selected upper blocks and output layers | Full model and activations, but gradients/optimizer state only for unfrozen weights | Complete changed checkpoint | A middle update budget; layer choice becomes a training decision |
| LoRA | Small correction matrices beside selected frozen layers | Full frozen base plus activations; optimizer state only for adapters | Small adapter paired with its base | Minimal trainable state and swappable task artifacts |
| QLoRA path | The same small correction matrices | A compact low-bit frozen base plus floating-point adapters and activations | Adapter plus a compatible low-bit base/runtime contract | Ordinary LoRA's trainable budget with lower frozen-base storage |

This table does **not** rank quality. More trainable state gives more freedom, but extra freedom can help, do nothing, or overfit. Quality requires matched data, budgets, seeds, and evaluation.

### The Two Axes Combine

Part 1 changed the learning experience. Part 2 changes where the resulting update can be stored. Any behavior objective could, in principle, use any parameter strategy:

| Behavior goal | Example parameter choices |
| --- | --- |
| Learn Riverside prose | Full FT, partial freezing, LoRA, or QLoRA |
| Follow editor instructions | Full FT, partial freezing, LoRA, or QLoRA |
| Prefer editor-ranked answers | Full FT, partial freezing, LoRA, or QLoRA |

This notebook repeatedly uses continued pretraining because it gives the parameter discussion one familiar behavioral target. The actual teaching runs differ in some data slices and hyperparameters, so only their update structure and parameter counts are directly comparable.

### Concept 4 (Parameter-Based): Full Fine-Tuning

Begin with the unconstrained reference: every weight can change. If the Aria Voss continuation produces an error, the optimizer may distribute the correction anywhere in the model.

That freedom makes full fine-tuning a useful baseline for the parameter axis. It also creates the largest training state, saves a complete changed model, and gives a small corpus the most opportunity to disturb general behavior.

Part 1 already produced this artifact at `./checkpoints/non-instruction-full`. The next cell does not retrain it; it measures what “allow every weight to move” means for the loaded SmolLM2 model.

**What the count can tell us:** the scale of trainable state relative to later strategies.

**What it cannot tell us:** whether full fine-tuning produced better Riverside behavior. That requires the fair comparison developed in Part 3.

> **PyTorch → Keras:** `p.numel()` counts elements in each parameter tensor and `p.requires_grad` flags whether it receives gradient updates; summing over `model.parameters()` gives total vs. trainable parameter counts. **Keras/TF equivalent:** `model.count_params()` gives total parameters directly, and the trainable subset is `sum(np.prod(w.shape) for w in model.trainable_weights)` — Keras tracks trainable/non-trainable via each layer's `trainable` attribute rather than a per-tensor `requires_grad` flag.

In [ ]:
# Load a throwaway copy of the base model purely to count its parameters
param_check_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Every parameter tensor's element count contributes to the total
total = sum(p.numel() for p in param_check_model.parameters())

# With nothing frozen yet, every parameter is also trainable
trainable = sum(p.numel() for p in param_check_model.parameters() if p.requires_grad)
print(
    f"Full fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.1f}%)"
)

# Free the memory now that the counts are captured
del param_check_model


### Concept 5 (Parameter-Based): Partial Fine-Tuning (Layer Freezing)

**Riverside's question for this section:** if full fine-tuning is the "maximum quality, maximum
laptop-fan-noise" option, is there a middle ground that still lets us re-train every time the catalog
grows, without waiting hours?

**The observation:** In transformer models, **early layers** learn general language features
(tokenization, basic syntax, common words) while **later layers** learn task-specific patterns. This
is similar to how early layers in CNNs detect edges, while later layers detect objects.

**The strategy:** Freeze everything, then selectively unfreeze:

- The last N transformer blocks (task-specific adaptation)
- The output head (final projection to vocabulary)

**Pros:**

- Much cheaper than full fine-tuning (only 10-30% of parameters)
- Less prone to catastrophic forgetting (general features preserved)
- No new architecture needed

**Cons:**

- Still edits raw model weights (can't easily "swap" like an adapter) -- so Riverside couldn't keep
  one editing-assistant checkpoint and one knowledge-base checkpoint without storing two full copies
  of `HuggingFaceTB/SmolLM2-135M-Instruct`
- Choosing _how many_ layers to unfreeze is a manual hyperparameter
- Middle ground: not as cheap as LoRA, not as powerful as full fine-tuning

**Example:** SmolLM2-135M has 30 transformer blocks. The runtime-derived policy below unfreezes the
last ~25% (7 blocks) plus the output head.


### Visualizing Layer-by-Layer Freezing

Before we run the code, let's visualize **exactly which layers** in `HuggingFaceTB/SmolLM2-135M-Instruct`
(30 transformer blocks) will be frozen vs. trainable when we unfreeze the last ~25% of blocks.

**The intuition:**

Think of the transformer as a **semantic refinement pipeline**:

| Layer                         | What it learns                                    | Freeze or Train? | Why?                                             |
| ----------------------------- | ------------------------------------------------- | ---------------- | ------------------------------------------------ |
| **Early blocks (~first 50%)** | Basic syntax, common words, tokenization patterns | FROZEN           | These are universal -- no need to change         |
| **Middle blocks (~50-75%)**   | Mid-level semantics, phrase structure             | FROZEN           | Still mostly general-purpose                     |
| **Last ~25% of blocks**       | Task-specific patterns, domain adaptation         | TRAINABLE        | This is where domain/task specialization happens |
| **Output head**               | Final vocabulary distribution                     | TRAINABLE        | Must learn domain-specific words                 |

**Why this works:**

1. **Early layers = general features:** Just like CNNs learn edges in early layers, transformer early
   blocks learn general language structure that's useful for _any_ task.
2. **Late layers = task-specific:** The final blocks learn task/domain-specific patterns. By only
   training these, we adapt to our corpus without forgetting general English.
3. **Catastrophic forgetting prevention:** Freezing ~75% of the model preserves general language
   ability while allowing focused adaptation.


In [ ]:
# Visualize partial freezing: which layers are trainable?
from transformers import AutoConfig
from matplotlib.patches import Patch, Rectangle

# Read the block count straight off the model config instead of hardcoding it
_freeze_cfg = AutoConfig.from_pretrained(MODEL_NAME)
n_layers = _freeze_cfg.num_hidden_layers
unfreeze_from = n_layers - max(2, n_layers // 4)  # unfreeze the last ~25% of blocks
layers = [f"Block {i}" for i in range(n_layers)] + ["Output Head"]
layer_positions = np.arange(len(layers))
tick_stride = max(1, n_layers // 12)  # keep y-axis labels readable regardless of depth

# Two side-by-side panels: frozen/trainable split, and relative gradient magnitude
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(6, n_layers * 0.3)))

# Plot 1: Frozen vs Trainable blocks. With 24+ rows there isn't room for per-bar text labels
# without them overlapping, so color + a legend carries the FROZEN/TRAINABLE distinction instead.
colors = ["lightblue" if i < unfreeze_from else "coral" for i in range(n_layers)] + [
    "coral"
]

# Draw one horizontal bar per block/head, colored by frozen vs. trainable
ax1.barh(
    layer_positions, [1] * len(layers), color=colors, edgecolor="black", linewidth=1.0
)

# Label every tick-stride-th row to keep the axis readable at any depth
ax1.set_yticks(layer_positions[::tick_stride])
ax1.set_yticklabels([layers[i] for i in layer_positions[::tick_stride]])
ax1.set_xlim(0, 1)
ax1.set_xticks([])
ax1.set_title(
    f"Partial Fine-Tuning Strategy\n({MODEL_NAME}: {n_layers} blocks)",
    fontsize=12,
    fontweight="bold",
)

# Put block 0 at the top of the chart instead of the bottom
ax1.invert_yaxis()

# Legend carries the FROZEN/TRAINABLE color meaning since bars have no room for text
ax1.legend(
    handles=[
        Patch(facecolor="lightblue", edgecolor="black", label="FROZEN"),
        Patch(facecolor="coral", edgecolor="black", label="TRAINABLE"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    fontsize=9,
)

# Plot 2: Gradient flow visualization
# Frozen blocks get zero gradient magnitude; trainable blocks ramp up toward the output head
gradient_flow = (
    [0.0] * unfreeze_from
    + list(np.linspace(0.5, 1.0, n_layers - unfreeze_from))
    + [1.0]
)

# Draw the relative gradient magnitude for every block/head as a horizontal bar
ax2.barh(
    layer_positions,
    gradient_flow,
    color="green",
    alpha=0.7,
    edgecolor="black",
    linewidth=1.0,
    label="Relative gradient magnitude",
)
ax2.set_yticks(layer_positions[::tick_stride])
ax2.set_yticklabels([layers[i] for i in layer_positions[::tick_stride]])
ax2.set_xlabel("Gradient Magnitude (relative)", fontsize=10)
ax2.set_title("Gradient Flow During Backpropagation", fontsize=12, fontweight="bold")
ax2.invert_yaxis()
ax2.set_xlim(0, 1.1)

# Midpoints of the frozen/trainable spans, used to place the two annotation labels below
frozen_mid = unfreeze_from / 2
trainable_mid = unfreeze_from + (n_layers - unfreeze_from) / 2

# Annotate the frozen span with "no gradient flow" text
ax2.text(
    0.05,
    frozen_mid,
    "No gradient\nflow",
    ha="center",
    va="center",
    fontsize=9,
    color="gray",
    fontweight="bold",
    style="italic",
)

# Annotate the trainable span with "full gradient flow" text
ax2.text(
    0.75,
    trainable_mid,
    "Full gradient\nflow",
    ha="center",
    va="center",
    fontsize=9,
    color="darkgreen",
    fontweight="bold",
)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), fontsize=9)

plt.tight_layout()
plt.show()

# Calculate parameter breakdown
total_blocks = n_layers
frozen_blocks = unfreeze_from
trainable_blocks = total_blocks - frozen_blocks
frozen_pct = (frozen_blocks / total_blocks) * 100
trainable_pct = 100 - frozen_pct

print(f"\n{'=' * 70}")
print(f"Partial Fine-Tuning Configuration:")
print(f"{'=' * 70}")
print(f"  Total blocks:      {total_blocks}")
print(
    f"  Frozen blocks:     {frozen_blocks} (blocks 0-{frozen_blocks-1}) — {frozen_pct:.1f}%"
)
print(
    f"  Trainable blocks:  {trainable_blocks} (blocks {unfreeze_from}-{total_blocks-1}) — {trainable_pct:.1f}%"
)
print(f"  Output head:       TRAINABLE")
print(f"{'=' * 70}")
print(f"  Memory savings:    ~{frozen_pct:.0f}% less optimizer state")
print(f"  Forgetting risk:   LOW (general features preserved)")
print(f"  Adaptation power:  MEDIUM (targeted domain learning)")
print(f"{'=' * 70}")


> **PyTorch → Keras:** setting `param.requires_grad = False` on every parameter freezes the entire model so no gradients flow to it during backprop. **Keras/TF equivalent:** `layer.trainable = False` on each layer (or `model.trainable = False` for the whole model) before compiling — Keras freezes at layer granularity, not per-tensor, and the change only takes effect after the model is (re)compiled.

In [ ]:
# Load a fresh base model instance to selectively freeze/unfreeze
freeze_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# Start every parameter frozen; the next cell re-enables gradients on the trainable slice
for param in freeze_model.parameters():
    param.requires_grad = False


### Selectively Unfreezing the Last ~25% of Blocks

Every parameter starts frozen above. This cell re-enables `requires_grad` on only the last few
transformer blocks (`unfreeze_from` onward) plus the final layer norm and output head -- the exact
split visualized earlier in this section -- so the optimizer only ever sees gradients for that
trainable slice.


> **PyTorch → Keras:** `model.named_parameters()` yields `(name, tensor)` pairs so individual parameters can be selectively re-enabled (`param.requires_grad = True`) by matching name substrings like block index or layer-norm/head names. **Keras/TF equivalent:** iterate `model.layers` and set `layer.trainable = True` for the specific layers you want to unfreeze (matched by `layer.name`), then recompile the model — Keras' selective-unfreezing story works the same way but at whole-layer granularity rather than per-parameter-tensor.

In [ ]:
n_layers = freeze_model.config.num_hidden_layers
unfreeze_from = n_layers - max(2, n_layers // 4)

# SmolLM2 decoder blocks are exposed directly as model.layers.
decoder_layers = freeze_model.model.layers
assert len(decoder_layers) == n_layers
for layer in decoder_layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.model.norm.parameters():
    parameter.requires_grad = True
for parameter in freeze_model.lm_head.parameters():
    parameter.requires_grad = True

# Verify the named-parameter prefixes agree with the direct module selection.
trainable_layer_prefixes = tuple(
    f"model.layers.{layer_index}." for layer_index in range(unfreeze_from, n_layers)
)
for name, parameter in freeze_model.named_parameters():
    if name.startswith(trainable_layer_prefixes):
        assert parameter.requires_grad, f"Expected trainable SmolLM2 layer parameter: {name}"

trainable = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in freeze_model.parameters())
print(
    f"Partial fine-tuning: {trainable:,}/{total:,} parameters trainable "
    f"({trainable / total * 100:.2f}%)"
)


### Building the Dataset and Running the Trainer

Same `tokenize_causal()`/`Trainer` pattern as continued pretraining, on a different 2-genre slice of
the corpus, with `learning_rate=1e-4` -- between full fine-tuning's `5e-5` and LoRA's `2e-4`, since
partial freezing updates more parameters than LoRA but far fewer than full fine-tuning.


> **PyTorch → Keras:** `Dataset`/`TrainingArguments`/`Trainer.train()` is HF's high-level PyTorch training loop (batching, optimizer, logging all handled internally), and `save_pretrained()` writes the model and config to disk. **Keras/TF equivalent:** `tf.data.Dataset` for batching, `model.compile(optimizer=..., loss=...)` to configure training, `model.fit(dataset, epochs=...)` to run it, and `model.save(...)` / `model.save_weights(...)` to persist — Keras' `fit()` plays the same role as `Trainer.train()`.

In [ ]:
# Partial freezing changes which parameters learn, not the causal-language-modeling objective.
# Load every chapter from two Riverside genres so later chapters are represented in training.

freeze_dataset = Dataset.from_dict(
    {"text": load_corpus_paragraphs(novels=["fantasy", "cyberpunk"])}
)

# Convert raw paragraphs into fixed-length input IDs, attention masks, and next-token labels.
# The same tokenize_causal() preprocessing is used for every parameter strategy so the
# comparison isolates the effect of freezing weights rather than changing the training data.

freeze_tokenized = freeze_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

# Configure the optimization budget for the unfrozen parameter slice.

training_args_freeze = TrainingArguments(
    # Store this strategy separately because partial freezing produces a complete model checkpoint.

    output_dir="./checkpoints/partial-freeze",
    # Process two tokenized sequences in each forward/backward pass.

    per_device_train_batch_size=2,
    # Bound the demonstration to 60 optimizer updates for a predictable runtime.

    max_steps=60,
    # Surface the training loss every ten steps so adaptation can be monitored.

    logging_steps=10,
    # Skip intermediate snapshots; the final trained model is saved explicitly below.

    save_strategy="no",
    # Use a middle-ground learning rate: higher than full fine-tuning, lower than LoRA,
    # because this strategy updates fewer parameters than full FT but many more than LoRA.

    learning_rate=1e-4,
    # Keep the run local instead of sending metrics to an external tracking service.

    report_to="none",
)

# Trainer receives the complete model, but its optimizer updates only tensors with
# requires_grad=True. Earlier cells left that flag enabled only for the final ~25% of
# transformer blocks, the final layer norm, and the output head.

trainer_freeze = Trainer(
    model=freeze_model, args=training_args_freeze, train_dataset=freeze_tokenized
)

# Each step still runs a forward pass through every layer. During backpropagation, frozen
# layers retain their pretrained weights while gradients update only the unfrozen upper slice.

trainer_freeze.train()

# Unlike LoRA, partial freezing edits weights in the original architecture rather than storing
# a small adapter, so save the complete fine-tuned weights and config for later reloading.

freeze_model.save_pretrained("./checkpoints/partial-freeze")
print("Saved partial (layer-freezing) fine-tune checkpoint.")

### Concept 6 (Parameter-Based): Parameter-Efficient Fine-Tuning (LoRA)

**Riverside's question for this section:** full and partial fine-tuning save a complete changed model for every job. Can Riverside keep one shared base and save only each job's small learned correction?

### Start with the Forward-Pass Intuition

Take the same Riverside continuation used throughout this notebook. Inside an attention layer, the base model already transforms the current activation into a useful language representation. Fine-tuning does not need to relearn that entire transformation if the Riverside-specific change is comparatively narrow.

LoRA keeps the original layer frozen and adds a second path:

1. the frozen base path produces the original layer output;
2. a small trainable path compresses the activation to a narrow bottleneck;
3. it expands that bottleneck back to the layer's output size;
4. the two outputs are added.

![LoRA parameter efficiency: a rank-4 adapter for a 20-input, 10-output layer](images/lora-low-rank-adaptation.png)

In the diagram, the input first passes through the small matrix $A$, producing four bottleneck values. Matrix $B$ expands those four values into an adjustment with the same shape as the frozen layer output. Training changes only $A$ and $B$.

The notation for that picture is intentionally compact:

$$
\text{layer output} = Wx + B(Ax).
$$

$W$ is the frozen base transformation. The adapter's effective weight correction is $\Delta W = BA$. The word **rank** refers to the bottleneck width: a smaller rank permits fewer independent directions of change but needs fewer trainable values.

### Make the Saving Concrete

For one square SmolLM2 attention projection with width $576$:

- changing the whole projection would expose $576 \times 576 = 331{,}776$ weights to training;
- rank $8$ uses two thin matrices with $8 \times 576 + 576 \times 8 = 9{,}216$ trainable weights;
- the original $331{,}776$ weights remain frozen and reusable.

The arithmetic is not the reason LoRA works; it expresses the physical idea shown above. Riverside stores one base model and small job-specific corrections rather than duplicating the full base for every assistant.

### What the Main Choices Mean

- **Rank `r`:** width of the bottleneck. More width gives the correction more freedom and increases memory.
- **Target modules:** which transformations receive a correction path. This notebook adapts attention projections.
- **Scale (`lora_alpha`):** how strongly the adapter output contributes relative to its rank.

These choices define capacity, not guaranteed quality. A higher rank can fit more change, but only matched training and evaluation can show whether the extra capacity helps Riverside's workload.

### What This Notebook Will Demonstrate

The next cells attach rank-8 adapters to the continued-pretraining objective, train them, inspect the actual injected matrices, and trace a real forward pass. That demonstrates where the update lives and how little state is trainable.

It does **not** prove that LoRA matches full fine-tuning in quality. The teaching runs use different data slices and budgets. Part 3 treats their outputs as hypotheses and explains the matched study needed for a quality claim.

**Operational advantages:** small optimizer state, swappable adapters, and the option to merge a chosen adapter into the base for serving.

**Operational constraints:** the base revision and tokenizer must remain compatible with the adapter, and the rank, scale, and target modules become versioned training choices.

This section applies LoRA to continued pretraining, not instruction tuning, to keep the behavior goal familiar while the location of learning changes.

> **PyTorch → Keras:** `LoraConfig(...)` declares the LoRA hyperparameters (rank, target modules, alpha), `get_peft_model(base, config)` wraps the frozen base model with trainable low-rank adapter matrices injected into the named target modules, and `print_trainable_parameters()` reports the resulting trainable/total ratio. **Keras/TF equivalent:** no first-party Keras LoRA API exists — the closest honest analog is manually freezing most layers (`layer.trainable = False`) and, for true low-rank adapters, hand-writing a custom `keras.layers.Layer` that adds a `BA` low-rank branch alongside a frozen dense layer; there's no drop-in `get_peft_model` equivalent.

In [ ]:
# SmolLM2 exposes separate query, key, value, and output projections in every attention block.
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
lora_config_pt = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
)

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = get_peft_model(lora_pt_base, lora_config_pt)
lora_pt_model.print_trainable_parameters()


### Building the Dataset

This run uses every chapter from three selected genres, following Part 1's policy for real training datasets. The novel list determines corpus breadth, while context length is set later by `tokenize_causal(max_length=128)`.

The selected genres differ from the partial-freezing run. That is acceptable for demonstrating the LoRA mechanics and measuring trainable parameters, but it does **not** form a controlled quality comparison between parameter strategies. Part 3 performs model selection under a shared evaluation context.

In [ ]:
# Load every qualifying paragraph from all chapters in the three selected genres.
# The novel list controls which domains enter this LoRA continued-pretraining run.
lora_pt_paragraphs = load_corpus_paragraphs(
    novels=["mystery", "horror", "literary"]
)

# Dataset.from_dict expects one column-oriented list: each paragraph becomes one raw row.
# This genre selection differs from the partial-freezing run, so it demonstrates LoRA's
# training mechanics; only parameter counts, not resulting model quality, are directly
# comparable across those runs.
lora_pt_dataset = Dataset.from_dict({"text": lora_pt_paragraphs})

# batched=True passes {"text": list[str]} into tokenize_causal(). Long paragraphs may
# produce multiple 128-token rows, while remove_columns drops the raw text after creating
# input_ids, attention_mask, and labels for causal next-token prediction.
lora_pt_tokenized = lora_pt_dataset.map(
    lambda examples: tokenize_causal(examples, tokenizer),
    batched=True,
    remove_columns=["text"],
)

# The later Trainer still stops after max_steps=60, so the shuffled run does not guarantee
# that one training pass visits every tokenized row in this full selected-novel corpus.
print(
    f"LoRA corpus: {len(lora_pt_paragraphs):,} paragraphs -> "
    f"{len(lora_pt_tokenized):,} fixed-length training chunks"
)

### Training and Saving

Same `2e-4` LoRA learning rate as instruction tuning, same `Trainer` pattern as every training cell in
this notebook -- this is the last of the five checkpoints this notebook trains before the head-to-head
comparison further down.


> **PyTorch → Keras:** same `TrainingArguments`/`Trainer.train()`/`save_pretrained()` pattern as the earlier training cell, here training only the LoRA adapter's parameters (the base stays frozen) and saving just the small adapter weights. **Keras/TF equivalent:** `model.compile(...)` + `model.fit(...)` with only the adapter sublayer's `trainable = True`, then `model.save_weights(...)` — since Keras has no adapter abstraction, this would typically save the whole model rather than a separate small adapter file.

In [ ]:
# Configure the training run: LoRA checkpoint dir, batch size, steps, higher LR than full FT
training_args_lora_pt = TrainingArguments(
    output_dir="./checkpoints/peft-lora",
    per_device_train_batch_size=2,
    max_steps=60,  # short instructional run; increase for a real convergence study
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

# Wire the LoRA-wrapped model, its training args, and the tokenized dataset together
trainer_lora_pt = Trainer(
    model=lora_pt_model, args=training_args_lora_pt, train_dataset=lora_pt_tokenized
)

# Run training -- only the LoRA adapter matrices actually receive gradient updates
trainer_lora_pt.train()

# Persist just the small adapter weights (the frozen base isn't re-saved)
lora_pt_model.save_pretrained("./checkpoints/peft-lora")
print("Saved parameter-efficient (LoRA) continued-pretraining adapter.")


### LoRA Decomposition: A Visual Intuition

Let's visualize exactly **what LoRA does** with concrete matrix dimensions. We'll use one real SmolLM2
attention projection to make it clear.

**Scenario:** SmolLM2-135M's query-projection matrix `W_q` is 576x576 (331,776 parameters).

**Full fine-tuning** would update: `W_new = W_old + ΔW` where ΔW is also 576x576 (331,776
trainable parameters).

**LoRA** instead uses: `W_new = W_old + B·A` where:

- `B` is 576x8 (4,608 parameters)
- `A` is 8x576 (4,608 parameters)
- **Total:** 9,216 trainable parameters (2.78% of the original)

**Key insight:** The rank bottleneck (`r=8`) forces the update to live in a low-dimensional subspace.
This is like saying "all the adaptation you need can be expressed as 8 basis vectors" instead of the
full 576-dimensional freedom.

**What does "rank" actually mean, and what is "intrinsic rank"?**

The **rank** of a matrix is how many genuinely independent rows (or columns) it has. A 576x576 matrix
can be stored as 331,776 numbers, but if every row is a combination of only 8 independent rows, its
rank is 8. `B(Ax)` can only produce rank-8 updates: `A` squeezes the input to 8 values and `B` expands
those values back to 576 dimensions.

That is a design choice backed by an empirical result. [Aghajanyan et al. 2020](https://arxiv.org/abs/2012.13255)
showed that pretrained language models have a surprisingly low **intrinsic dimension**: a model can
adapt well while moving in a small subspace of its full parameter space. LoRA's authors
([Hu et al. 2021](https://arxiv.org/abs/2106.09685)) applied that idea to the weight update `ΔW` itself.
The pretrained model already contains broad language knowledge, so a concentrated low-rank correction
can be enough for a specific task.

**Why does this work?**

1. **Task adaptations are low-rank:** Fine-tuning for a specific task does not need to change every
   direction in the 576-dimensional space.
2. **Overfitting resistance:** Fewer trainable parameters reduce the capacity to memorize a small dataset.
3. **Efficient gradient flow:** The low-rank bottleneck regularizes updates toward a compact set of directions.


In [ ]:
# Visualize LoRA matrix decomposition
from matplotlib.patches import Patch, Rectangle

# Four side-by-side panels: full ΔW, A, B, and the resulting low-rank B(Ax)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

d, r = base_model.config.hidden_size, 8  # SmolLM2 hidden size, read dynamically

# Plot 1: Full fine-tuning ΔW
# Draw ΔW as one big d×d square to represent every trainable parameter
axes[0].add_patch(Rectangle((0, 0), d, d, fill=True, color="steelblue", alpha=0.6))
axes[0].set_xlim(0, d)
axes[0].set_ylim(0, d)
axes[0].set_aspect("equal")
axes[0].set_title(
    f"Full Fine-Tuning: ΔW\n{d}×{d} = {d*d:,} trainable params", fontsize=11
)
axes[0].set_xlabel(f"{d}")
axes[0].set_ylabel(f"{d}")

# Overlay the exact parameter count in the center of the square
axes[0].text(
    d / 2,
    d / 2,
    f"{d*d:,}\nparameters",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="white",
)
axes[0].legend(
    handles=[
        Patch(
            facecolor="steelblue", alpha=0.6, edgecolor="black", label="Full ΔW (dense)"
        )
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

# Plot 2: LoRA matrix A -- hits the input FIRST, projects d dims DOWN to r dims
# Draw A as a wide, short rectangle to represent its d×r shape
axes[1].add_patch(Rectangle((0, 0), d, r, fill=True, color="mediumseagreen", alpha=0.7))
axes[1].set_xlim(0, d)
axes[1].set_ylim(0, r + 100)
axes[1].set_aspect("equal")
axes[1].set_title(
    f"LoRA Matrix A (down-project)\n{r}×{d} = {r*d:,} params", fontsize=11
)
axes[1].set_xlabel(f"{d}")
axes[1].set_ylabel(f"{r}")

# Overlay A's parameter count in the center of the rectangle
axes[1].text(
    d / 2,
    r / 2,
    f"{r*d:,}\nparams",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)
axes[1].legend(
    handles=[
        Patch(
            facecolor="mediumseagreen",
            alpha=0.7,
            edgecolor="black",
            label="A (down-project, trainable)",
        )
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

# Plot 3: LoRA matrix B -- takes A's r-dim output and projects it back UP to d dims
# Draw B as a narrow, tall rectangle to represent its d×r shape
axes[2].add_patch(Rectangle((0, 0), r, d, fill=True, color="coral", alpha=0.7))
axes[2].set_xlim(0, r + 100)
axes[2].set_ylim(0, d)
axes[2].set_aspect("equal")
axes[2].set_title(f"LoRA Matrix B (up-project)\n{d}×{r} = {d*r:,} params", fontsize=11)
axes[2].set_xlabel(f"{r}")
axes[2].set_ylabel(f"{d}")

# Overlay B's parameter count in the center of the rectangle
axes[2].text(
    r / 2,
    d / 2,
    f"{d*r:,}\nparams",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)
axes[2].legend(
    handles=[
        Patch(
            facecolor="coral",
            alpha=0.7,
            edgecolor="black",
            label="B (up-project, trainable)",
        )
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

# Plot 4: B(Ax) result (low-rank approximation)
# Draw the resulting update as a full d×d square, same size as ΔW but rank-limited
axes[3].add_patch(Rectangle((0, 0), d, d, fill=True, color="mediumpurple", alpha=0.6))
axes[3].set_xlim(0, d)
axes[3].set_ylim(0, d)
axes[3].set_aspect("equal")
axes[3].set_title(
    f"B(Ax) = Low-Rank ΔW\n{d}×{d} with rank {r}\nTotal: {d*r + r*d:,} params",
    fontsize=11,
)
axes[3].set_xlabel(f"{d}")
axes[3].set_ylabel(f"{d}")

# Label the square with the rank of the update it can produce
axes[3].text(
    d / 2,
    d / 2,
    f"rank-{r}\nupdate",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="white",
)

# Draw a double-headed arrow to emphasize the constrained degrees of freedom
axes[3].annotate(
    "",
    xy=(d / 2, d - 50),
    xytext=(d / 2, 50),
    arrowprops=dict(arrowstyle="<->", lw=2, color="yellow"),
)
axes[3].text(
    d / 2 + 80,
    d / 2,
    f"Only {r} degrees\nof freedom!",
    fontsize=9,
    color="yellow",
    fontweight="bold",
)
axes[3].legend(
    handles=[
        Patch(
            facecolor="mediumpurple",
            alpha=0.6,
            edgecolor="black",
            label="B(A(x)) low-rank ΔW",
        )
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

# Hide the numeric tick marks on every panel since the shapes are already labeled
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

# Print parameter savings
full_params = d * d
lora_params = d * r + r * d
saving_pct = (1 - lora_params / full_params) * 100

print(f"\n{'=' * 70}")
print(f"Parameter Efficiency Analysis (d={d}, r={r}):")
print(f"{'=' * 70}")
print(f"  Full fine-tuning:  {full_params:>10,} parameters (100.0%)")
print(
    f"  LoRA adaptation:   {lora_params:>10,} parameters ({lora_params/full_params*100:>5.2f}%)"
)
print(
    f"  Savings:           {full_params - lora_params:>10,} parameters ({saving_pct:>5.2f}%)"
)
print(f"{'=' * 70}")
print(f"  Memory savings: ~{saving_pct:.1f}% less optimizer state")
print(f"  Training speed: ~{saving_pct:.1f}% fewer gradients to compute")
print(f"  Swappability: Can load/unload adapters in <1 second")
print(f"{'=' * 70}")


### Crack Open the Adapter We Just Trained

The diagram and equation described a generic low-rank correction. Now follow the continued-pretraining adapter from the preceding cells into a real SmolLM2 attention layer.

We will inspect `lora_pt_model`, the adapter just trained on Riverside prose, rather than switching to Part 1's instruction adapter. That keeps one question intact: **where did the continued-pretraining update live when the base stayed frozen?**

The next cells locate PEFT's injected matrices, verify their shapes and rank limit, and capture one raw Riverside continuation as it passes through both the frozen query projection and the learned LoRA correction.

> **PyTorch → Keras:** `model.named_modules()` walks the full module tree so code can find every submodule PEFT injected (checking for a `lora_A` attribute), then reads the frozen `base_layer` and trainable `lora_A`/`lora_B` matrices directly off the object. **Keras/TF equivalent:** `model.layers` (recursively via nested `submodules`) walks the layer graph, and each layer's `layer.weights`/`get_weights()` exposes its tensors — Keras has no PEFT-style wrapper object, so there's no `lora_A`/`lora_B` attribute to introspect unless you built the adapter yourself as a custom layer.

In [ ]:
# Find every SmolLM2 projection wrapped by the continued-pretraining LoRA adapter.
lora_layers = [
    (name, module)
    for name, module in lora_pt_model.named_modules()
    if hasattr(module, "lora_A") and len(getattr(module, "lora_A")) > 0
]
expected_adapters = lora_pt_model.config.num_hidden_layers * len(LORA_TARGET_MODULES)
print(
    f"PEFT wrapped {len(lora_layers)} projections; "
    f"expected {expected_adapters} for {len(LORA_TARGET_MODULES)} targets per decoder layer."
)
print("First wrapped module names:")
for name, _ in lora_layers[:8]:
    print(" ", name)

# Use the first query projection for concrete shape and rank introspection.
name0, layer0 = next(
    (name, module) for name, module in lora_layers if name.endswith("q_proj")
)
base0 = layer0.base_layer
lora_A0 = layer0.lora_A["default"]
lora_B0 = layer0.lora_B["default"]
scaling0 = layer0.scaling["default"]

print()
print(f"Inside {name0}:")
print(f"  Frozen q_proj:      {type(base0).__name__}, weight {tuple(base0.weight.shape)}")
print(f"  lora_A (down-proj): {tuple(lora_A0.weight.shape)}  <- trainable")
print(f"  lora_B (up-proj):   {tuple(lora_B0.weight.shape)}  <- trainable")
print(f"  scaling (alpha/r):  {scaling0}")
print(f"  Trained lora_B norm: {lora_B0.weight.norm().item():.4f}")

### Verifying the Rank Claim on the Real Trained Adapter

`lora_A0`/`lora_B0` above are real, trained matrices -- so the "rank ≤ r" claim from the intuition
section isn't just a shape argument, it's checkable. Build the actual effective update this adapter
contributes, `ΔW = scaling · B·A`, and look at its singular values: by construction, no more than `r`
of them can be non-zero, no matter what values training found for `A` and `B`. This is the real
mechanics behind "low-rank update," measured on the exact adapter trained earlier in this notebook --
not a toy shape diagram.


> **PyTorch → Keras:** `layer.get_delta_weight(...)` (a PEFT method) computes the effective weight update `scaling · B @ A` as a plain tensor, and `torch.linalg.svdvals(...)` computes its singular values to verify the rank constraint. **Keras/TF equivalent:** `tf.linalg.svd(matrix, compute_uv=False)` computes singular values the same way; since Keras has no PEFT wrapper, you'd first need to manually multiply your own `B`/`A` weight matrices (`tf.matmul(B, A)`) to get the delta before taking its SVD.

In [ ]:
# Verify LoRA's rank constraint on the real trained adapter -- not asserted, measured via SVD.
delta_W = (
    layer0.get_delta_weight("default").detach().cpu()
)  # PEFT's own scaling * B @ A computation

# Singular values reveal how many independent directions ΔW actually has
singular_values = torch.linalg.svdvals(delta_W)
r = lora_A0.weight.shape[
    0
]  # the configured rank, read directly off the trained A matrix

# Only the first few dozen singular values are ever non-negligible for a rank-r update -- plotting
# all min(delta_W.shape) of them would squeeze the real cliff into an invisible sliver, so zoom in
# and use a log y-axis, which makes an 8-orders-of-magnitude drop actually visible.
n_show = min(30, len(singular_values))
floor = 1e-8  # log scale needs a positive floor; true near-zero values are clipped up for display only

# Clip near-zero singular values up to the floor so the log-scale axis can still plot them
plot_values = np.clip(singular_values[:n_show].numpy(), floor, None)

fig, ax = plt.subplots(figsize=(9, 4.5))

# Color the first r bars (real degrees of freedom) differently from the rest
ax.bar(
    range(n_show),
    plot_values,
    color=["mediumseagreen" if i < r else "lightgray" for i in range(n_show)],
)
ax.set_yscale("log")
ax.set_xlabel(f"Singular value index (first {n_show} of {len(singular_values)})")
ax.set_ylabel("Singular value magnitude (log scale)")
ax.set_title(
    f"Singular Value Spectrum of the Real Trained \u0394W = scaling \u00b7 B\u00b7A\n"
    f"({tuple(delta_W.shape)} matrix, configured rank r={r})",
    fontsize=11,
    fontweight="bold",
)
ax.legend(
    handles=[
        Patch(
            facecolor="mediumseagreen",
            edgecolor="black",
            label=f"First {r} singular values (the adapter's real degrees of freedom)",
        ),
        Patch(
            facecolor="lightgray",
            edgecolor="black",
            label=f"Indices {r + 1}-{n_show} (~0 by construction, floored for the log scale)",
        ),
    ],
    fontsize=8,
)
plt.tight_layout()
plt.show()

# Count how many singular values are meaningfully non-zero
nonzero = (singular_values > 1e-4).sum().item()
print(
    f"\u0394W shape: {tuple(delta_W.shape)} -> full rank would allow up to {min(delta_W.shape)} singular values"
)
print(f"Singular values above 1e-4: {nonzero} (matches the configured rank r={r})")
print(
    f"Largest singular value: {singular_values[0].item():.4f}   {r}th singular value: {singular_values[r - 1].item():.4f}"
)
print(
    f"First value past the rank cutoff (index {r}): {singular_values[r].item():.2e}  <- effectively zero"
)
print(
    "This is the real mechanics behind 'low-rank update': it isn't that training happened to find a "
    "low-rank \u0394W -- B(A(x))'s construction makes it mathematically impossible for \u0394W to have "
    "more than r independent directions, no matter what A and B learn."
)


### Tracing a Real Forward Pass Through the Adapted Layer

`lora_B` started at all-zeros, so before training this adapter was a mathematical no-op: the combined
output was _exactly_ the frozen base output. After training, `lora_B` has real values (norm printed
above), so now it nudges the output. Let's actually capture both paths -- the frozen base output and
the full (base + LoRA) output -- for a real prompt, using forward hooks (no reimplementing the math by
hand, so there's no risk of getting the internal computation subtly wrong).


> **PyTorch → Keras:** `module.register_forward_hook(...)` attaches a callback that captures a layer's output tensor during the forward pass, and the model call runs inside `torch.no_grad()` since no training is happening. **Keras/TF equivalent:** the closest analog is building an auxiliary `keras.Model` whose outputs include the intermediate layer(s) you want (`keras.Model(inputs=model.input, outputs=[layer.output, model.output])`), since Keras has no direct hook API; gradient tracking is simply avoided by not using a `tf.GradientTape()`.

In [ ]:
# Capture one real SmolLM2 query projection before and after its LoRA correction.
from matplotlib.patches import Patch

captured = {}


def make_hook(key):
    def hook(module, inputs, output):
        captured[key] = output.detach().cpu()
    return hook


hook_base = base0.register_forward_hook(make_hook("base_only"))
hook_combined = layer0.register_forward_hook(make_hook("combined"))
demo_prompt = STORY_SEED
enc_lora = tokenizer(demo_prompt, return_tensors="pt").to(device)
lora_pt_model.eval()
with torch.no_grad():
    _ = lora_pt_model(**enc_lora)
hook_base.remove()
hook_combined.remove()

base_out = captured["base_only"][0]
combined_out = captured["combined"][0]
lora_delta = combined_out - base_out
last_pos = base_out.shape[0] - 1
show_dims = min(60, base_out.shape[-1])

print(f"Raw continuation prompt: {demo_prompt!r}")
print(
    f"q_proj output shape: {tuple(base_out.shape)}; "
    "SmolLM2 uses separate q_proj/k_proj/v_proj/o_proj modules"
)
print()
print("At the last token position:")
print(f"  ||base q_proj output|| = {base_out[last_pos].norm().item():.3f}")
print(f"  ||LoRA delta||         = {lora_delta[last_pos].norm().item():.5f}")
print(
    "  delta / base norm     = "
    f"{(lora_delta[last_pos].norm() / base_out[last_pos].norm()).item():.4%}"
)

fig_static, (ax_static1, ax_static2) = plt.subplots(1, 2, figsize=(14, 4))
ax_static1.plot(
    base_out[last_pos, :show_dims].numpy(),
    color="steelblue",
    label="frozen q_proj",
)
ax_static1.plot(
    combined_out[last_pos, :show_dims].numpy(),
    color="coral",
    linestyle="--",
    label="q_proj + LoRA",
)
ax_static1.set_title(
    f"SmolLM2 Query Projection - first {show_dims} dimensions", fontsize=11
)
ax_static1.set_xlabel("q_proj output dimension")
ax_static1.legend(fontsize=8)
ax_static1.grid(alpha=0.3)

ax_static2.bar(
    np.arange(show_dims),
    lora_delta[last_pos, :show_dims].numpy(),
    color="mediumseagreen",
    label="LoRA delta",
)
ax_static2.set_title(
    f"LoRA delta at token position {last_pos}: scaling * B(A(x))", fontsize=11
)
ax_static2.set_xlabel("q_proj output dimension")
ax_static2.axhline(0, color="black", linewidth=0.8)
ax_static2.legend(fontsize=8)
ax_static2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_tokens = lora_delta.shape[0]
delta_arr = lora_delta[:, :show_dims].numpy()
delta_max = np.abs(delta_arr).max() * 1.15 or 1e-6
fig_anim, ax_anim = plt.subplots(figsize=(12, 4))
bar_colors = ["mediumseagreen" if value >= 0 else "coral" for value in delta_arr[0]]
bars_anim = ax_anim.bar(np.arange(show_dims), delta_arr[0], color=bar_colors)
ax_anim.axhline(0, color="black", linewidth=0.8)
ax_anim.set_ylim(-delta_max, delta_max)
ax_anim.set_xlim(-1, show_dims)
ax_anim.set_xlabel(f"q_proj output dimension (first {show_dims})")
ax_anim.set_ylabel("LoRA delta")
tokens_decoded = tokenizer.convert_ids_to_tokens(enc_lora["input_ids"][0].tolist())
title_obj = ax_anim.set_title("")


def _update(frame):
    deltas = delta_arr[frame]
    for bar, value in zip(bars_anim, deltas):
        bar.set_height(value)
        bar.set_color("mediumseagreen" if value >= 0 else "coral")
    title_obj.set_text(
        f"SmolLM2 q_proj LoRA delta - token {frame}/{n_tokens - 1} "
        f"{tokens_decoded[frame]!r}; ||delta|| = {np.linalg.norm(deltas):.5f}"
    )
    return list(bars_anim) + [title_obj]


anim = FuncAnimation(fig_anim, _update, frames=n_tokens, interval=160, blit=False)
plt.close(fig_anim)
display(HTML(anim.to_jshtml(fps=6)))

### Adapter Portability: Swapping Onto a Compatible Base

LoRA adapters are swappable because they store small deltas rather than another complete base model. Compatibility is stricter than matching one matrix width: the adapter must use the same base architecture, target-module names, tensor shapes, tokenizer contract, and preferably the exact base revision used for training.

A freshly instantiated `HuggingFaceTB/SmolLM2-135M-Instruct` base is compatible with adapters regenerated by Parts 1 and 2 for that same identifier. A checkpoint produced for another architecture is not compatible, even when a few dimensions happen to match. PEFT validates the configured targets and tensor shapes during loading.

The code below loads the continued-pretraining adapter, swaps in Part 1's instruction adapter on the same resident SmolLM2 base, and inspects the actual query-projection adapter shape.


> **PyTorch → Keras:** `PeftModel.from_pretrained(...)` attaches one adapter to a base model, `load_adapter(...)` registers a second adapter on the same base without reloading its weights, and `set_adapter(...)` switches which adapter's deltas are active — all cheap pointer/dict operations inside PEFT's routing layer. **Keras/TF equivalent:** no built-in adapter-swapping mechanism exists; the practical substitute is keeping separate fully fine-tuned (or separately frozen/unfrozen) model copies and calling `model.load_weights(...)` to switch between them, which is far more expensive than PEFT's adapter swap since it reloads full weight sets rather than a tiny delta.

In [ ]:
import gc
import os
from peft import PeftModel

swap_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
swap_model = PeftModel.from_pretrained(swap_base, "./checkpoints/peft-lora")
swap_model.eval()

adapter_kb = sum(
    os.path.getsize(os.path.join("./checkpoints/peft-lora", filename))
    for filename in os.listdir("./checkpoints/peft-lora")
) / 1024
base_mb = sum(
    parameter.numel() * parameter.element_size()
    for parameter in swap_base.parameters()
) / 1024**2
print(f"Base model in memory: {base_mb:.0f} MB (shared across adapters)")
print(f"Continued-pretraining adapter on disk: {adapter_kb:.0f} KB")
print()
print("[Continued-pretraining adapter]")
print(f"  {generate(swap_model, STORY_SEED, instruction=False)}")

swap_model.load_adapter("./checkpoints/instruction-lora", adapter_name="instruct")
swap_model.set_adapter("instruct")
print()
print("[Instruction adapter]")
print(f"  {generate(swap_model, PROMPT)}")

swap_model.set_adapter("default")
print()
print("[Continued-pretraining adapter, restored]")
print(f"  {generate(swap_model, STORY_SEED, instruction=False)}")

q_proj_adapter = next(
    module
    for name, module in swap_model.named_modules()
    if name.endswith("q_proj")
    and hasattr(module, "lora_A")
    and "default" in module.lora_A
)
q_proj_A_shape = tuple(q_proj_adapter.lora_A["default"].weight.shape)
print()
print(f"q_proj LoRA A shape: {q_proj_A_shape}")
print(
    f"Compatibility contract: base={MODEL_NAME}, "
    f"hidden_size={swap_model.config.hidden_size}, targets={LORA_TARGET_MODULES}"
)
print(
    "Adapters from the previous architecture must be regenerated; "
    "checkpoint files are not converted in place."
)

del swap_base, swap_model
gc.collect()


### Concept 7 (Parameter-Based): QLoRA - Put the Frozen Base on a Memory Diet

Ordinary LoRA solved one problem: very little state receives gradients or optimizer history. It did **not** make the frozen base disappear.

Return to Riverside's continued-pretraining adapter. The adapter may be tiny, yet every forward pass still needs the full base model in memory. For SmolLM2 that is manageable. For a multi-billion-parameter model, the frozen state can be the reason training does not fit on one GPU.

The next question is therefore:

> If the base will never be updated, can it be stored more compactly while the small LoRA path remains trainable?

That combination is **QLoRA**: keep the base frozen in a low-bit representation and train floating-point LoRA matrices beside it.

![QLoRA architecture showing a frozen 4-bit NF4 base dequantized to bf16 for computation alongside trainable bf16 LoRA adapters](images/qlora-quantized-base-lora-adapters.png)

Read the diagram as two paths through the same layer:

1. **Frozen base path:** compact codes and small scale values represent the base weights in memory. The runtime reconstructs the needed weight values into a compute-friendly type for matrix multiplication.
2. **Trainable adapter path:** LoRA $A$ and $B$ remain ordinary floating-point parameters, receive gradients, and produce the learned correction.
3. **Addition:** the base output and adapter correction are added exactly as in ordinary LoRA.

The common 4-bit format used for the frozen base is called **NF4**. Its implementation is blockwise: nearby weights share scale metadata, which lets small codes represent local ranges more accurately than one scale for the entire model. The important intuition is not the codebook formula; it is that compact storage and arithmetic precision are separate choices. The base can be stored in four-bit codes without asking the matrix multiplication itself to operate as crude four-bit arithmetic.

```mermaid
flowchart LR
    X["Activation"] --> BASE["Reconstruct frozen base weights<br/>for this computation"]
    CODES["Compact frozen codes<br/>+ scales"] --> BASE
    X --> A["Trainable LoRA A"] --> B["Trainable LoRA B"]
    BASE --> ADD(("Add"))
    B --> ADD
    ADD --> Y["Layer output"]
```

During backpropagation, gradients may pass through the base computation so earlier activations receive useful signals, but the frozen codes do not update. Optimizer state is needed only for the LoRA matrices.

### What the CPU Exercise Can and Cannot Show

The next cell uses a tiny uniform four-bit approximation because this notebook is designed to run without a CUDA-specific QLoRA stack. It demonstrates the structure:

- compact frozen codes are reconstructed for computation;
- a floating-point LoRA branch is added;
- gradients land only on the adapter.

It is **not** an NF4 implementation, a GPU memory benchmark, a trained QLoRA checkpoint, or evidence about QLoRA quality. A real run needs supported GPU kernels and libraries such as bitsandbytes, then the same matched quality evaluation required for every other parameter strategy.

SmolLM2 is small enough that ordinary LoRA is the honest choice for this local run. QLoRA becomes useful when the frozen base, not the adapter, is the memory bottleneck.

The deeper kernel and format details belong in [Quantization in Depth](../../ai-infrastructure/06-quantization/quantization-in-depth.ipynb#appendix-b-nf4-and-qlora), after this storage-versus-compute intuition is secure.

In [ ]:
# CPU-only structural analogy: uniform 4-bit codes, not bitsandbytes NF4.
torch.manual_seed(7)
in_features, out_features, rank = 4, 3, 2
alpha = 4

activation = torch.randn(2, in_features)
base_weight = torch.randn(out_features, in_features)  # frozen reference weights

# Approximate each output row with signed 4-bit codes and one scale.
scale = base_weight.abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 7
quantized_codes = torch.clamp(torch.round(base_weight / scale), -8, 7).to(torch.int8)
reconstructed_weight = quantized_codes.float() * scale

# Only these small floating-point matrices are trainable.
lora_a = torch.nn.Parameter(torch.randn(rank, in_features) * 0.05)
lora_b = torch.nn.Parameter(torch.randn(out_features, rank) * 0.05)

base_output = activation @ reconstructed_weight.T
adapter_output = (activation @ lora_a.T) @ lora_b.T * (alpha / rank)
layer_output = base_output + adapter_output
layer_output.square().mean().backward()

print(f"Stored base codes: {quantized_codes.dtype}; reconstructed: {reconstructed_weight.dtype}")
print(f"Base is frozen: {base_weight.requires_grad is False}")
print(f"LoRA gradients: A={lora_a.grad.norm():.4f}, B={lora_b.grad.norm():.4f}")
print(f"Toy reconstruction MAE: {(base_weight - reconstructed_weight).abs().mean():.4f}")


### Optional Production Extension: Shrink the Artifact After Training

The parameter-strategy story so far asked what must fit in memory **during training**. Riverside also has a later question: after choosing a trained candidate, can the artifact be stored and served more compactly?

This is not another fine-tuning method. It is a conversion performed after training. Keep the same local example in mind: Riverside has a continued-pretraining checkpoint that can complete the Aria Voss passage, and now wants a smaller CPU artifact without changing that behavior unacceptably.

The next experiment performs real PyTorch dynamic int8 conversion on that checkpoint and checks three observable consequences:

1. How much smaller is the serialized model state?
2. Does the same prompt still produce a plausible continuation?
3. Did fit to a wholly excluded Riverside novel deteriorate enough to reject the conversion?

The third question uses perplexity only as a before/after regression check on the **same** model and evaluation text. Part 3 develops the intuition behind that summary; this section does not use it as a universal quality score.

The excluded `historical` novel was not used by the three-novel continued-pretraining demo. If Riverside retrained on all novels, it would have to reserve another approved corpus before training; a test set cannot be invented after the model has already seen everything.

### What This Experiment Cannot Decide

Dynamic int8 is one CPU runtime path, not QLoRA training and not a universal serving recommendation. A production decision also needs measurements on the target hardware, representative request lengths and concurrency, supported kernels, latency, memory, cost, and workload-specific quality checks. Those conditions cannot be reproduced honestly by one local notebook process.

The broader method catalog belongs in [Quantization in Depth](../../ai-infrastructure/06-quantization/quantization-in-depth.ipynb). Stay here for the narrow Riverside conversion experiment, or skip to the parameter-count comparison if the training story is your focus.

> **PyTorch → Keras:** `torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` converts every `nn.Linear` layer's weights to int8 (with activations quantized on-the-fly at inference), and `model.state_dict()` saved via `torch.save` measures the resulting size. **Keras/TF equivalent:** TensorFlow's native dynamic-range quantization goes through the TFLite converter — `tf.lite.TFLiteConverter.from_keras_model(model)` with `converter.optimizations = [tf.lite.Optimize.DEFAULT]` — which produces a separate `.tflite` file rather than quantizing the in-memory Keras model in place.

In [ ]:
# Real, CPU-native post-training quantization. Compare one trained checkpoint before and after
# conversion; this evaluates deployment quantization, not QLoRA training.
import io
import math

# Reload the Part 1 continued-pretraining demo checkpoint at full fp32 precision.
full_checkpoint_dir = Path("./checkpoints/non-instruction-full")
full_checkpoint_config = json.loads(
    (full_checkpoint_dir / "config.json").read_text(encoding="utf-8")
)
if full_checkpoint_config.get("model_type") != base_model.config.model_type:
    raise RuntimeError(
        "The continued-pretraining checkpoint uses the previous architecture. "
        "Regenerate Part 1 checkpoints for MODEL_NAME."
    )
fp32_model = AutoModelForCausalLM.from_pretrained(full_checkpoint_dir)
fp32_model.eval()

# Dynamic quantization stores supported Linear weights as int8 and quantizes activations at runtime.
int8_model = torch.quantization.quantize_dynamic(
    fp32_model, {torch.nn.Linear}, dtype=torch.qint8
)


def state_dict_size_mb(model):
    """Measure serialized parameter-state size without writing a temporary artifact."""
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / 1e6


def quick_perplexity(model, paragraphs, max_length=128):
    """Compute perplexity on one shared paragraph sample."""
    if not paragraphs:
        raise ValueError("Evaluation requires at least one paragraph.")

    model.eval()
    losses = []
    with torch.inference_mode():
        for paragraph in paragraphs:
            encoded = tokenizer(
                paragraph, truncation=True, max_length=max_length, return_tensors="pt"
            )
            output = model(**encoded, labels=encoded["input_ids"])
            losses.append(output.loss.item())
    return math.exp(sum(losses) / len(losses))


def generate_cpu(model, prompt, max_new_tokens=40):
    """Generate with CPU-resident fp32 or dynamically quantized models."""
    model.eval()
    formatted_prompt = format_instruction(prompt)
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        generated[0][prompt_length:], skip_special_tokens=True
    ).strip()


fp32_mb = state_dict_size_mb(fp32_model)
int8_mb = state_dict_size_mb(int8_model)
print(f"fp32 state_dict size: {fp32_mb:.1f} MB")
print(f"int8 state_dict size: {int8_mb:.1f} MB")
print(f"Real size reduction: {(1 - int8_mb / fp32_mb) * 100:.1f}%")

# The demo checkpoint was trained on every chapter of these three novels in Part 1.
CONTINUED_PRETRAINING_NOVELS = {"scifi", "fantasy", "mystery"}
HELD_OUT_NOVEL = "historical"
assert HELD_OUT_NOVEL not in CONTINUED_PRETRAINING_NOVELS

# Reserve a different novel altogether, then select a deterministic spread for a quick CPU check.
held_out_novel_paragraphs = load_corpus_paragraphs(novels=[HELD_OUT_NOVEL])
EVAL_PARAGRAPH_COUNT = 32
sample_stride = max(1, len(held_out_novel_paragraphs) // EVAL_PARAGRAPH_COUNT)
quant_holdout = held_out_novel_paragraphs[::sample_stride][:EVAL_PARAGRAPH_COUNT]
print(
    f"Evaluation split: {len(quant_holdout)} paragraphs from the wholly excluded "
    f"'{HELD_OUT_NOVEL}' novel"
)

fp32_ppl = quick_perplexity(fp32_model, quant_holdout)
int8_ppl = quick_perplexity(int8_model, quant_holdout)
print(f"\nHeld-out-novel perplexity, fp32: {fp32_ppl:.1f}")
print(f"Held-out-novel perplexity, int8: {int8_ppl:.1f}")
print(f"Perplexity delta from conversion: {int8_ppl - fp32_ppl:+.2f}")

print("\n=== Same prompt, both precisions ===")
print(f"fp32: {generate_cpu(fp32_model, PROMPT)}")
print(f"int8: {generate_cpu(int8_model, PROMPT)}")

print(
    "\nRiverside's takeaway: dynamic int8 quantization is a post-training deployment lever. "
    "Accept it only when the size, latency, and held-out quality changes fit the target runtime."
)


### Visual Comparison: Parameter Counts Across All Techniques

Concepts 4-6 have now each trained and saved a real checkpoint (full fine-tuning, partial freezing,
LoRA). Let's close out the parameter axis by visualizing **exactly how much memory/compute each
approach required**, using the real trained models rather than hypothetical numbers -- this makes
concrete why LoRA has become the industry standard.

> **Where's QLoRA on this chart?** Nowhere -- deliberately. QLoRA's _trainable_-parameter count would
> land at roughly the same tiny bar as LoRA (same adapter, same rank); what it additionally saves is
> frozen _base-weight_ memory, a dimension this chart doesn't plot, and it was never trained in this
> run (GPU-specific, per Concept 7). Adding a bar for a number we didn't measure would be exactly the
> kind of fabricated chart this notebook avoids elsewhere.


> **PyTorch → Keras:** aggregates trainable-parameter counts across the three already-trained models by summing `p.numel()` over `model.parameters()`, filtered by `p.requires_grad` where relevant, purely for the comparison chart. **Keras/TF equivalent:** `model.count_params()` for totals, and `sum(np.prod(w.shape) for w in model.trainable_weights)` for the trainable subset of each model being compared — the same aggregation, just reading Keras' trainable-weights list instead of PyTorch's per-tensor flag.

In [ ]:
# Visual parameter comparison across all techniques

# Real parameter counts pulled from the actual models trained earlier in this notebook
# (not hardcoded, so this stays correct no matter which base model MODEL_NAME points to)
total_params = sum(p.numel() for p in base_model.parameters())
full_ft_params = total_params  # 100%
partial_ft_params = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)
lora_params = sum(p.numel() for p in lora_pt_model.parameters() if p.requires_grad)

techniques = ["Full\nFine-Tuning", "Partial\nFreezing", "LoRA"]
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]
colors = ["steelblue", "coral", "mediumseagreen"]

# Two side-by-side panels: absolute parameter counts and relative memory footprint
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: absolute trainable parameter counts
bars = ax1.bar(
    techniques, param_counts, color=colors, alpha=0.82, edgecolor="black", linewidth=1.2
)
ax1.set_yscale("log")
ax1.set_ylim(top=max(param_counts) * 2.2)
ax1.set_ylabel("Trainable parameters (log scale)", fontsize=12, fontweight="bold")
ax1.set_title(
    f"Absolute Parameter Counts ({MODEL_NAME})", fontsize=13, fontweight="bold", pad=14
)
ax1.grid(alpha=0.25, axis="y", linestyle="--")
ax1.set_axisbelow(True)
ax1.spines[["top", "right"]].set_visible(False)

for bar, count, pct in zip(bars, param_counts, param_pcts):
    ax1.annotate(
        f"{count:,}\n{pct:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

# Plot 2: estimated training memory, normalized to full fine-tuning
bytes_per_param = 4 + 8  # fp32 weight (4 bytes) + Adam optimizer state (8 bytes)
mem_gb = [count * bytes_per_param / 1e9 for count in param_counts]
memory_labels = [
    f"{mem_gb[0]:.2f} GB",
    f"{mem_gb[1] * 1000:.0f} MB",
    f"{mem_gb[2] * 1000:.1f} MB",
]
y_positions = np.arange(len(techniques))
horizontal_bars = ax2.barh(
    y_positions,
    param_pcts,
    color=colors,
    alpha=0.82,
    edgecolor="black",
    linewidth=1.2,
    height=0.58,
)
ax2.set_xscale("log")
ax2.set_xlim(0.1, 180)
ax2.set_yticks(y_positions, [name.replace("\n", " ") for name in techniques])
ax2.invert_yaxis()
ax2.set_xlabel("Memory relative to full fine-tuning (%)", fontsize=12, fontweight="bold")
ax2.set_title(
    "Relative Training Memory\n(trainable params + optimizer state)",
    fontsize=13,
    fontweight="bold",
    pad=10,
)
ax2.grid(False)
ax2.grid(alpha=0.25, axis="x", which="both", linestyle="--")
ax2.set_axisbelow(True)
ax2.spines[["top", "right", "left"]].set_visible(False)
ax2.tick_params(axis="y", length=0)

for bar, pct, memory in zip(horizontal_bars, param_pcts, memory_labels):
    ax2.annotate(
        f"{pct:.2f}%  |  ~{memory}",
        xy=(bar.get_width(), bar.get_y() + bar.get_height() / 2),
        xytext=(7, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="#222222",
    )

fig.tight_layout(w_pad=4)
plt.show()

# Print detailed breakdown
print(f"\n{'=' * 80}")
print(f"Memory & Compute Analysis for {MODEL_NAME} ({total_params / 1e6:.0f}M params):")
print(f"{'=' * 80}")
print(
    f"{'Technique':<20} {'Trainable':<15} {'%':<8} {'Memory Est.':<15} {'Training Speed'}"
)
print(f"{'-' * 80}")
print(
    f"{'Full Fine-Tuning':<20} {f'{full_ft_params:,}':<15} {f'{param_pcts[0]:.2f}%':<8} {f'~{mem_gb[0]:.2f} GB':<15} {'1.0x (baseline)'}"
)
print(
    f"{'Partial Freezing':<20} {f'{partial_ft_params:,}':<15} {f'{param_pcts[1]:.2f}%':<8} {f'~{mem_gb[1] * 1000:.0f} MB':<15} {f'~{total_params / max(partial_ft_params, 1):.1f}x faster'}"
)
print(
    f"{'LoRA (r=8)':<20} {f'{lora_params:,}':<15} {f'{param_pcts[2]:.2f}%':<8} {f'~{mem_gb[2] * 1000:.1f} MB':<15} {f'~{total_params / max(lora_params, 1):.0f}x faster'}"
)
print(f"{'-' * 80}")
print()
print("For a 70B model, these same ratios apply and the differences become MASSIVE:")
print("  • Full FT: ~840 GB → requires 8×A100 80GB GPUs")
print("  • LoRA:    ~3 GB → fits on a single consumer GPU (RTX 4090)")
print(f"{'=' * 80}")

---

## Checkpoint Inventory Before Packaging

Two more trained artifacts now join the candidates from Part 1:

| Checkpoint on disk | What changed | Question it helps answer |
| --- | --- | --- |
| `./checkpoints/partial-freeze` | Only the upper portion of the model remained trainable | Is a middle-cost parameter strategy competitive? |
| `./checkpoints/peft-lora` | Continued pretraining used a small LoRA adapter | Is the cheapest swappable strategy good enough? |

At this point the training roadmap has produced six checkpoint objects in total: the untouched baseline plus five fine-tuned candidates. It has also demonstrated post-training quantization and explained the QLoRA path without pretending that a seventh QLoRA checkpoint was trained.

One question remains deliberately unanswered: **which artifact should Riverside deploy for which workload?** Answering it requires reloading the candidates under one evaluation context and separating data-objective effects from parameter-strategy effects. Part 3 performs that comparison after this notebook finishes the production artifact and packaging boundary below.

---

## Same Prompts, Four Parameter Strategies

This recap reloads the baseline, full fine-tuning, partial-freezing, and LoRA checkpoints produced across Parts 1 and 2. Each candidate receives the same raw continuation prompts with greedy decoding, and only one model is resident at a time.

Treat the table as a **visual diagnostic, not a controlled ranking**. The teaching runs used different genre slices and learning rates, so output differences combine parameter strategy with data exposure. A defensible ranking requires matched training data, seeds, token budgets, and held-out evaluation.


In [ ]:
import gc
import html
import time

from IPython.display import HTML, display
from peft import PeftModel


PARAMETER_COMPARISON_PROMPTS = {
    "Sci-fi continuation": "Aria Voss checked the Meridian's Promise status panel and",
    "Fantasy continuation": "Kerra Valmont felt all five tides simultaneously and",
    "Mystery continuation": "Elena Voss studied the 1879 survey map and realized",
}

PARAMETER_COMPARISON_CANDIDATES = [
    ("Base", "base", MODEL_NAME),
    ("Full fine-tuning", "full", "./checkpoints/non-instruction-full"),
    ("Partial freezing", "full", "./checkpoints/partial-freeze"),
    ("LoRA", "adapter", "./checkpoints/peft-lora"),
]


def load_parameter_comparison_candidate(kind, model_path):
    """Load one parameter-strategy candidate and leave the others on disk."""
    if kind == "base":
        return AutoModelForCausalLM.from_pretrained(model_path).to(device)

    artifact_path = Path(model_path)
    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Missing {artifact_path}. Run the corresponding training cell before this comparison."
        )
    if kind == "full":
        return AutoModelForCausalLM.from_pretrained(artifact_path).to(device)

    adapter_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    return PeftModel.from_pretrained(adapter_base, artifact_path).to(device)


def generate_parameter_comparison_answer(model, prompt, max_new_tokens=48):
    """Generate a deterministic raw continuation for a matched visual comparison."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        generated[0][prompt_length:], skip_special_tokens=True
    ).strip() or "[model stopped immediately]"


parameter_comparison_answers = {
    prompt_label: {} for prompt_label in PARAMETER_COMPARISON_PROMPTS
}
for candidate_label, candidate_kind, candidate_path in PARAMETER_COMPARISON_CANDIDATES:
    candidate_model = load_parameter_comparison_candidate(candidate_kind, candidate_path)
    try:
        for prompt_label, prompt in PARAMETER_COMPARISON_PROMPTS.items():
            started = time.perf_counter()
            answer = generate_parameter_comparison_answer(candidate_model, prompt)
            elapsed = time.perf_counter() - started
            parameter_comparison_answers[prompt_label][candidate_label] = (
                f"{answer}\n\n[{elapsed:.1f}s]"
            )
    finally:
        del candidate_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

header = "".join(
    f"<th style='min-width:190px'>{html.escape(label)}</th>"
    for label, _, _ in PARAMETER_COMPARISON_CANDIDATES
)
body = "".join(
    "<tr>"
    f"<th style='text-align:left;vertical-align:top'>{html.escape(prompt_label)}</th>"
    + "".join(
        "<td style='vertical-align:top;white-space:pre-wrap'>"
        f"{html.escape(parameter_comparison_answers[prompt_label][label])}</td>"
        for label, _, _ in PARAMETER_COMPARISON_CANDIDATES
    )
    + "</tr>"
    for prompt_label in PARAMETER_COMPARISON_PROMPTS
)
display(
    HTML(
        "<table><thead><tr><th>Same prompt</th>"
        + header
        + "</tr></thead><tbody>"
        + body
        + "</tbody></table>"
    )
)


---

## Production Deployment Pattern

Suppose later evaluation selects Part 2's continued-pretraining LoRA candidate for Riverside's house-style continuation workload. The training directory is not yet a release. A release must let another process reconstruct exactly what was tested.

Follow that one adapter through the production boundary:

1. **Pin the base:** record the exact SmolLM2 revision and tokenizer that give the adapter its meaning.
2. **Package the adapter:** copy its config and weights into a new versioned directory rather than mutating the training output.
3. **Record file digests:** later loading can detect a missing or changed file before serving.
4. **Reconstruct and smoke-test:** load the pinned base plus adapter and run the same Riverside continuation contract once to catch packaging failures.
5. **Evaluate before traffic:** packaging success says the artifact is loadable, not that it is good enough to release.
6. **Release gradually:** send a small share of real requests to the candidate while the accepted version remains available.
7. **Recover quickly:** route requests back to the accepted version if live quality or service behavior degrades.

| Technique | What must be packaged | How serving reconstructs the candidate |
| --- | --- | --- |
| Full fine-tuning / partial freezing | Complete model weights, config, and tokenizer reference | Load one self-contained checkpoint per release |
| LoRA | Adapter config and weights plus the exact base-model and tokenizer revisions | Keep the base resident and load the small adapter; merge only after separately validating the merged artifact |
| QLoRA path | LoRA adapter plus an explicitly compatible low-bit base and runtime configuration | Reconstruct the quantized base on supported hardware, then attach the adapter |

The code below models only the **package, verify, load, and smoke-test boundaries** on the local filesystem. It remains disabled because publishing requires an exact base revision and an intentional output location.

It cannot demonstrate production readiness. This notebook has no independent release benchmark, live editor traffic, concurrency load, service monitoring, or accepted rollback artifact. Part 3 supplies the decision structure; a real service must supply the versioned evidence and operational environment.

**Practical controls**

1. Keep training, evaluation, packaging, and deployment as separate retryable stages.
2. Never place credentials or raw private prompts in notebook source, manifests, logs, or checkpoints.
3. Record release ID, base revision, adapter name, request latency, failures, token counts, memory, and approved quality summaries without logging raw manuscripts by default.
4. Retain the previous immutable release so recovery changes routing rather than rebuilding a model under pressure.
5. Measure serving cost directly: small trainable state reduces training and storage costs, but the resident base, precision, sequence lengths, batching, and replica count dominate inference cost.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil


@dataclass(frozen=True)
class ProductionReleaseConfig:
    """Configuration for promoting one notebook checkpoint into an immutable release."""

    release_id: str = "riverside-peft-v1"
    strategy: str = "lora"  # full, partial-freeze, lora, or qlora
    source_dir: Path = Path("./checkpoints/peft-lora")
    release_root: Path = Path("./production-artifacts")
    base_model_id: str = MODEL_NAME
    base_revision: str = "SET_EXACT_BASE_REVISION"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Return a streaming SHA-256 digest without loading a large checkpoint into RAM."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def required_artifact_groups(strategy: str) -> tuple[tuple[str, ...], ...]:
    """Return alternative required filenames for the notebook's parameter strategies."""
    if strategy in {"lora", "qlora"}:
        return (
            ("adapter_config.json",),
            ("adapter_model.safetensors", "adapter_model.bin"),
        )
    if strategy in {"full", "partial-freeze"}:
        return (
            ("config.json",),
            ("model.safetensors", "pytorch_model.bin"),
        )
    raise ValueError(f"Unsupported strategy: {strategy}")


def validate_artifact_dir(source_dir: Path, strategy: str) -> None:
    """Fail before publication when a checkpoint is incomplete or mislabeled."""
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Checkpoint directory does not exist: {source_dir}")
    for alternatives in required_artifact_groups(strategy):
        if not any((source_dir / name).is_file() for name in alternatives):
            raise FileNotFoundError(
                f"Expected one of {alternatives} in {source_dir} for strategy={strategy!r}"
            )


def package_model_release(config: ProductionReleaseConfig) -> Path:
    """Copy, checksum, and atomically publish a versioned local model release."""
    if config.base_revision == "SET_EXACT_BASE_REVISION":
        raise ValueError("Pin base_revision to the exact tested model commit before publishing.")
    validate_artifact_dir(config.source_dir, config.strategy)

    config.release_root.mkdir(parents=True, exist_ok=True)
    final_dir = config.release_root / config.release_id
    staging_dir = config.release_root / f".{config.release_id}.staging"
    if final_dir.exists() or staging_dir.exists():
        raise FileExistsError(
            f"Release or staging directory already exists for {config.release_id!r}"
        )

    try:
        artifact_dir = staging_dir / "model"
        shutil.copytree(config.source_dir, artifact_dir)
        files = {
            str(path.relative_to(staging_dir)): {
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in sorted(artifact_dir.rglob("*"))
            if path.is_file()
        }
        manifest = {
            "schema_version": 1,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "release": {
                **asdict(config),
                "source_dir": str(config.source_dir),
                "release_root": str(config.release_root),
            },
            "files": files,
        }
        (staging_dir / "manifest.json").write_text(
            json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
        )
        os.replace(staging_dir, final_dir)  # Atomic when staging and release share a filesystem.
        return final_dir
    except Exception:
        shutil.rmtree(staging_dir, ignore_errors=True)
        raise


PRODUCTION_RELEASE = ProductionReleaseConfig()
RUN_PRODUCTION_PACKAGE = False

if RUN_PRODUCTION_PACKAGE:
    published_dir = package_model_release(PRODUCTION_RELEASE)
    print(f"Published immutable release: {published_dir.resolve()}")
else:
    print("Production packaging is configured but disabled; set RUN_PRODUCTION_PACKAGE = True to publish.")


In [ ]:
from dataclasses import dataclass
import json
from pathlib import Path
import time


@dataclass(frozen=True)
class ProductionServeConfig:
    """Configuration for loading and smoke-testing one promoted release."""

    release_dir: Path = Path("./production-artifacts/riverside-peft-v1")
    device: str = device
    max_new_tokens: int = 32


def verify_release(release_dir: Path) -> dict:
    """Load the manifest and verify every published file before model loading."""
    manifest_path = release_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for relative_path, expected in manifest["files"].items():
        artifact_path = release_dir / relative_path
        if not artifact_path.is_file():
            raise FileNotFoundError(f"Release artifact is missing: {artifact_path}")
        if artifact_path.stat().st_size != expected["bytes"]:
            raise ValueError(f"Release artifact size changed: {artifact_path}")
        if sha256_file(artifact_path) != expected["sha256"]:
            raise ValueError(f"Release artifact checksum failed: {artifact_path}")
    return manifest


def load_production_candidate(config: ProductionServeConfig):
    """Reconstruct a complete checkpoint or a PEFT adapter release for inference."""
    manifest = verify_release(config.release_dir)
    release = manifest["release"]
    strategy = release["strategy"]
    artifact_dir = config.release_dir / "model"
    base_model_id = release["base_model_id"]
    base_revision = release["base_revision"]

    serving_tokenizer = AutoTokenizer.from_pretrained(
        base_model_id, revision=base_revision
    )
    if serving_tokenizer.pad_token is None:
        serving_tokenizer.pad_token = serving_tokenizer.eos_token

    if strategy in {"lora", "qlora"}:
        if strategy == "qlora":
            if config.device != "cuda":
                raise RuntimeError("QLoRA serving requires the configured 4-bit CUDA runtime.")
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id,
                revision=base_revision,
                quantization_config=quantization_config,
                device_map="auto",
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id, revision=base_revision
            ).to(config.device)
        model = PeftModel.from_pretrained(base_model, artifact_dir)
    else:
        model = AutoModelForCausalLM.from_pretrained(artifact_dir).to(config.device)

    model.eval()
    return serving_tokenizer, model, manifest


def smoke_test_candidate(tokenizer, model, manifest: dict, config: ProductionServeConfig):
    """Run a deterministic probe and return observability fields without the raw prompt."""
    formatted_prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": PROMPT},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}

    started = time.perf_counter()
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=config.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    latency_ms = (time.perf_counter() - started) * 1000
    prompt_tokens = inputs["input_ids"].shape[1]
    output_tokens = generated.shape[1] - prompt_tokens
    completion = tokenizer.decode(
        generated[0][prompt_tokens:], skip_special_tokens=True
    ).strip()
    metrics = {
        "release_id": manifest["release"]["release_id"],
        "strategy": manifest["release"]["strategy"],
        "latency_ms": round(latency_ms, 1),
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
    }
    return completion, metrics


PRODUCTION_SERVE = ProductionServeConfig()
RUN_PRODUCTION_SMOKE_TEST = False

if RUN_PRODUCTION_SMOKE_TEST:
    production_tokenizer, production_model, production_manifest = load_production_candidate(
        PRODUCTION_SERVE
    )
    production_completion, production_metrics = smoke_test_candidate(
        production_tokenizer, production_model, production_manifest, PRODUCTION_SERVE
    )
    print(json.dumps(production_metrics, indent=2))
    print(f"Smoke-test completion: {production_completion}")
else:
    print("Production loading is configured but disabled; set RUN_PRODUCTION_SMOKE_TEST = True to run it.")


---

## Roadmap Checkpoint: From Training Choices to Model Selection

The two tuning axes are now ready to reconnect:

| Axis completed | What Riverside chose during training | What remains unknown |
| --- | --- | --- |
| Data objective (Part 1) | Continued pretraining, SFT, or DPO depending on the behavior to teach | Which behavior is required by each workload? |
| Parameter strategy (Part 2) | Full fine-tuning, partial freezing, LoRA, or the QLoRA path depending on resources | Which efficiency trade-off preserves enough quality? |
| Production artifact | Full checkpoint or versioned adapter paired with its base | Which candidate passes the relevant release evidence? |

Do not collapse those questions into one leaderboard. A model can be best at predicting Riverside prose and still be worse at following an editor's instruction.

Continue to **[Part 3: Comparison & Decision](03-llm-finetuning-comparison-and-decision.ipynb)**. Its opening roadmap shows where all seven concepts landed, then the notebook compares the measured candidates without treating objective differences as parameter-strategy results.